In [2]:
!pip install pandas sqlalchemy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
import pandas as pd
from sqlalchemy import create_engine

# Raw data with string inconsistencies, mixed units, and formatting errors
raw_mining_data = {
    'sample_id': ['S-001', 'sample_02', 'S.003', 'S-004', 'SAMPLE_05', 's006', 'S-007', 'S-008'],
    'ore_grade': ['0.55%', '0.12%', '0.88%', '450ppm', '1.20%', '0.05%', 'below_detection', '15.5%'],
    'site_zone': ['north-zone', 'NORTH_ZONE', 'North', 'south', 'South_Zone', 'south', 'NORTH', 'South'],
    'extraction_date': ['2023-01-10', '10/01/2023', '2023.01.11', '2023-01-12', '12-01-2023', '2023-01-13', '2023-01-14', '2023-01-15']
}

df_raw = pd.DataFrame(raw_mining_data)
engine = create_engine('sqlite://')
df_raw.to_sql('raw_ore_samples', engine, index=False)

print("Database 'raw_ore_samples' created and ready for SQL cleaning.")
print(df_raw.head(8))

Database 'raw_ore_samples' created and ready for SQL cleaning.
   sample_id        ore_grade   site_zone extraction_date
0      S-001            0.55%  north-zone      2023-01-10
1  sample_02            0.12%  NORTH_ZONE      10/01/2023
2      S.003            0.88%       North      2023.01.11
3      S-004           450ppm       south      2023-01-12
4  SAMPLE_05            1.20%  South_Zone      12-01-2023
5       s006            0.05%       south      2023-01-13
6      S-007  below_detection       NORTH      2023-01-14
7      S-008            15.5%       South      2023-01-15


In [4]:
# Query di anteprima - NON modifica la tabella originale
q_cleaning = """
SELECT 
    -- Pulizia sample_id
    CASE
        WHEN sample_id = 'sample_02' THEN 'S-002'
        WHEN sample_id = 'S.003' THEN 'S-003'
        WHEN sample_id = 'SAMPLE_05' THEN 'S-005'
        WHEN sample_id = 's006' THEN 'S-006'
        ELSE sample_id
    END AS sample_id,
    
    -- Pulizia site_zone
    CASE
        WHEN site_zone = 'north-zone' THEN 'North'
        WHEN site_zone = 'NORTH_ZONE' THEN 'North'
        WHEN site_zone = 'NORTH' THEN 'North'
        WHEN site_zone = 'south' THEN 'South'
        WHEN site_zone = 'South_Zone' THEN 'South'
        ELSE site_zone
    END AS site_zone,
    
    -- Pulizia ore_grade (ppm -> % e below_detection -> NULL)
    CASE
        WHEN ore_grade LIKE '%ppm' THEN 
            (CAST(REPLACE(ore_grade, 'ppm', '') AS FLOAT) / 10000) 
        WHEN ore_grade LIKE '%' AND ore_grade != 'below_detection' THEN 
        CAST(REPLACE(ore_grade, '%', '') AS FLOAT)
        WHEN ore_grade = 'below_detection' THEN NULL
        ELSE ore_grade
    END AS ore_grade,
    
    -- Pulizia date (formato unico YYYY-MM-DD)
    CASE
        -- Formato DD/MM/YYYY (10/01/2023)
        WHEN extraction_date LIKE '%/%' THEN 
            SUBSTR(extraction_date, 7, 4) || '-' ||
            SUBSTR(extraction_date, 4, 2) || '-' ||
            SUBSTR(extraction_date, 1, 2)
        -- Formato YYYY.MM.DD (2023.01.11)
        WHEN extraction_date LIKE '%.%' THEN 
            REPLACE(extraction_date, '.', '-')
        -- Formato DD-MM-YYYY (12-01-2023)
        WHEN extraction_date LIKE '%-%' AND SUBSTR(extraction_date, 1, 2) <= '31' THEN
            SUBSTR(extraction_date, 7, 4) || '-' ||
            SUBSTR(extraction_date, 4, 2) || '-' ||
            SUBSTR(extraction_date, 1, 2)
        ELSE extraction_date
    END AS extraction_date
FROM raw_ore_samples;
"""

# Esegui la query
df_cleaned = pd.read_sql_query(q_cleaning, engine)
df_cleaned.to_sql('clean_data', engine, index=False, if_exists='replace')
print("=" * 50)
print("Clean Data:")
print("=" * 50)
print(df_cleaned.to_string(index=False))

Clean Data:
sample_id site_zone  ore_grade extraction_date
    S-001     North      0.550      1-10-3--20
    S-002     North      0.120      2023-01-10
    S-003     North      0.880      2023-01-11
    S-004     South      0.045      1-12-3--20
    S-005     South      1.200      2023-01-12
    S-006     South      0.050      1-13-3--20
    S-007     North        NaN      1-14-3--20
    S-008     South     15.500      1-15-3--20


In [5]:
q_stats = """
SELECT 
    AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS media,
    MIN(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS minimo,
    MAX(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS massimo,
    MAX(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) - 
    MIN(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS range
FROM clean_data
WHERE ore_grade IS NOT NULL AND ore_grade != 'below_detection';
"""

df_stats = pd.read_sql_query(q_stats, engine)

print("=" * 50)
print("Measures:")
print("=" * 50)
print(df_stats.to_string(index=False))

Measures:
   media  minimo  massimo  range
2.620714   0.045     15.5 15.455


In [6]:
q_zonation_stats = """
SELECT 
    site_zone,
    AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS media,
    COUNT(*) AS campioni
FROM clean_data
GROUP BY site_zone;
"""
df_zonation_stats = pd.read_sql_query(q_zonation_stats, engine)

print("=" * 50)
print("Measures:")
print("=" * 50)
print(df_zonation_stats.to_string(index=False))

Measures:
site_zone    media  campioni
    North 0.516667         4
    South 4.198750         4


In [9]:
q_std_dev = """
SELECT 
    AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) AS media,
    SQRT(
        AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT) * CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) - 
        AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT)) * AVG(CAST(REPLACE(ore_grade, '%', '') AS FLOAT))
    ) AS deviazione_std
FROM clean_data
WHERE ore_grade IS NOT NULL AND ore_grade != 'below_detection';
"""
df_std_dev = pd.read_sql_query(q_std_dev, engine)

print("=" * 50)
print("Measures:")
print("=" * 50)
print(df_std_dev.to_string(index=False))

Measures:
   media  deviazione_std
2.620714        5.274045
